In [1]:
!gdown --folder 10d38UfG7DDRkl9LYf6vJQH2pcMWnMtwx -O Garberus_model

Retrieving folder contents
Processing file 1gS8gdTEqYFQSZevqg6FCBoWuKarfrYKP config.json
Processing file 1Eo9OhAlt_y8WOLyLJzIKXp9xeWLCJILR garberus.pth
Processing file 175JkjiIdIc38EYoVdvVEfppcIj9ImHwj preprocessor_config.json
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1gS8gdTEqYFQSZevqg6FCBoWuKarfrYKP
To: /content/Garberus_model/config.json
100% 34.4k/34.4k [00:00<00:00, 65.7MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Eo9OhAlt_y8WOLyLJzIKXp9xeWLCJILR
From (redirected): https://drive.google.com/uc?id=1Eo9OhAlt_y8WOLyLJzIKXp9xeWLCJILR&confirm=t&uuid=58acc635-3735-496e-89a5-5427513382ef
To: /content/Garberus_model/garberus.pth
100% 489M/489M [00:07<00:00, 62.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=175JkjiIdIc38EYoVdvVEfppcIj9ImHwj
To: /content/Garberus_model/preprocessor_config.json
100% 412/412 [00:00<00:00, 2.20MB/s]
Download com

In [2]:
%%writefile test.py
import os
import math
import argparse
from glob import glob
from tqdm import tqdm
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoFeatureExtractor, TimesformerConfig, TimesformerForVideoClassification

# ================= CONFIG =================
RESIZE = 224
NUM_FRAMES_CLIP = 4
CLIP_STRIDE = 1
BATCH_SIZE = 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CHECKPOINT_MULTIHEAD = "Garberus_model/garberus.pth"

THRESHOLD_DUMP_RATIO = 0.3     # soglia “dump video”


# ================= MODEL WRAPPER =================
class MultiHeadTimesformer(nn.Module):
    def __init__(self):
        super().__init__()
        config = TimesformerConfig.from_pretrained("Garberus_model/config.json")
        self.backbone = TimesformerForVideoClassification(config)

        embed_dim = self.backbone.classifier.out_features
        self.dump_head = nn.Linear(embed_dim, 2)
        self.timestamp_head = nn.Linear(embed_dim, 2)

    def _extract_feat(self, outputs):
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            return outputs.pooler_output
        if hasattr(outputs, "last_hidden_state") and outputs.last_hidden_state is not None:
            return outputs.last_hidden_state.mean(dim=1)
        return outputs.logits

    def forward(self, pixel_values):
        outputs = self.backbone(pixel_values, return_dict=True)
        feat = self._extract_feat(outputs)
        return self.dump_head(feat), self.timestamp_head(feat)


# ================= FRAME EXTRACTION =================
def extract_clip_frames(video_path, start_sec, num_frames=NUM_FRAMES_CLIP, resize=RESIZE):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    start_frame = int(start_sec * fps)
    frame_indices = [start_frame + int(i * (fps / num_frames)) for i in range(num_frames)]

    frames = []
    for fi in frame_indices:
        if fi >= total_frames or total_frames <= 0:
            frame = np.zeros((resize, resize, 3), dtype=np.uint8)
        else:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ok, frame = cap.read()
            if not ok:
                frame = np.zeros((resize, resize, 3), dtype=np.uint8)
            else:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frame = cv2.resize(frame, (resize, resize))

        frames.append(frame)

    cap.release()
    return frames


# ================= DATASET =================
class VideoClipDataset(Dataset):
    def __init__(self, video_paths, fe):
        self.samples = []
        self.fe = fe

        for vp in video_paths:
            cap = cv2.VideoCapture(vp)
            fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
            total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            duration = total_frames / fps if total_frames > 0 else 0.0
            cap.release()

            n_clips = int(math.ceil(duration))
            for sec in range(0, n_clips, CLIP_STRIDE):
                self.samples.append((vp, sec))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        vp, sec = self.samples[idx]
        frames = extract_clip_frames(vp, sec)
        pixel_values = self.fe(frames, return_tensors="pt")["pixel_values"]
        return {
            "pixel_values": pixel_values.squeeze(0),
            "video_path": vp,
            "start_sec": sec,
        }


def collate_fn(batch):
    pv = torch.stack([b["pixel_values"] for b in batch]).to(DEVICE)
    vp = [b["video_path"] for b in batch]
    sec = [b["start_sec"] for b in batch]
    return {"pixel_values": pv, "video_path": vp, "start_sec": sec}


# ================= INFERENCE =================
def infer_folder(videos_folder, results_folder):

    # ---- prende i video ----
    exts = ("*.mp4", "*.avi", "*.mov", "*.mkv")
    video_paths = sorted(sum([glob(os.path.join(videos_folder, e)) for e in exts], []))
    if len(video_paths) == 0:
        print("No videos found.")
        return

    os.makedirs(results_folder, exist_ok=True)

    fe = AutoFeatureExtractor.from_pretrained("Garberus_model", local_files_only=True)
    dataset = VideoClipDataset(video_paths, fe)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    model = MultiHeadTimesformer().to(DEVICE)
    model.load_state_dict(torch.load(CHECKPOINT_MULTIHEAD, map_location=DEVICE), strict=False)
    model.eval()

    # per-second outputs
    per_video_dump = {}
    per_video_ts = {}
    per_video_dump_probs = {}
    per_video_ts_probs = {}

    # final stats
    total_dump_videos = 0
    ts_effettivo = 0
    ts_centrale = 0

    print("\n--- INFERENCE ---\n")

    for batch in tqdm(loader, desc="Inferenza"):
        pv = batch["pixel_values"]

        dump_logits, ts_logits = model(pv)

        dump_preds = torch.argmax(dump_logits, dim=1).cpu().numpy()
        ts_preds   = torch.argmax(ts_logits, dim=1).cpu().numpy()

        dump_probs = torch.softmax(dump_logits, dim=1).detach().cpu().numpy()
        ts_probs   = torch.softmax(ts_logits, dim=1).detach().cpu().numpy()

        for i, vp in enumerate(batch["video_path"]):
            s = batch["start_sec"][i]

            per_video_dump.setdefault(vp, {})[s] = dump_preds[i]
            per_video_ts.setdefault(vp, {})[s] = ts_preds[i]

            per_video_dump_probs.setdefault(vp, {})[s] = dump_probs[i]
            per_video_ts_probs.setdefault(vp, {})[s] = ts_probs[i]

    # ================= POST-PROCESSING video-level =================
    print("\n--- OUTPUT PER VIDEO ---\n")

    dump_videos_means = []
    nodump_videos_means = []

    for vp in video_paths:
        fname = os.path.basename(vp)
        name_txt = os.path.splitext(fname)[0] + ".txt"
        txt_path = os.path.join(results_folder, name_txt)

        secs_sorted = sorted(per_video_dump[vp])
        votes = [per_video_dump[vp][s] for s in secs_sorted]

        dump_ratio = sum(votes) / len(votes)
        video_is_dump = (dump_ratio >= THRESHOLD_DUMP_RATIO)

        # ----- calcolo PROBABILITÀ MEDIE per video -----
        dump_mat = np.array([per_video_dump_probs[vp][s] for s in secs_sorted])
        ts_mat   = np.array([per_video_ts_probs[vp][s] for s in secs_sorted])

        dump_mean = dump_mat.mean(axis=0)
        ts_mean   = ts_mat.mean(axis=0)

        print(f"{fname}:")
        print(f"   dump_mean_0 = {dump_mean[0]:.4f}   dump_mean_1 = {dump_mean[1]:.4f}")
        print(f"   ts_mean_0   = {ts_mean[0]:.4f}   ts_mean_1   = {ts_mean[1]:.4f}")
        print(f"   dump_ratio = {dump_ratio:.3f}   -> video_dump = {int(video_is_dump)}\n")

        # accumulo per la media finale
        if video_is_dump:
            dump_videos_means.append(dump_mean)
        else:
            nodump_videos_means.append(dump_mean)

        # ----- timestamp -----
        if not video_is_dump:
            open(txt_path, "w").close()
            continue

        total_dump_videos += 1
        pred_secs = sorted([s for s in secs_sorted if per_video_ts[vp][s] == 1])

        if len(pred_secs) == 0:
            ts_centrale += 1
            center = int(math.floor(len(secs_sorted) / 2))
            with open(txt_path, "w") as f:
                f.write(str(center))
        else:
            ts_effettivo += 1
            first, last = pred_secs[0], pred_secs[-1]
            predicted = int(math.ceil(first + 0.5 * (last - first)))
            predicted = max(predicted, 0)
            with open(txt_path, "w") as f:
                f.write(str(predicted))


# ================= CLI =================
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--videos", required=True)
    parser.add_argument("--results", required=True)
    args = parser.parse_args()

    infer_folder(args.videos, args.results)
    print("Finished.")


Writing test.py


In [6]:
!python test.py --videos videos --results results

2025-12-08 13:24:54.558189: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765200294.592645    6284 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765200294.602280    6284 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1765200294.628363    6284 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765200294.628416    6284 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1765200294.628424    6284 computation_placer.cc:177] computation placer alr